# 关于invoke参数的使用

## 1、invoke的传参

### 文本输入
举例：

In [2]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

# 1、读取.env配置文件中的信息。相关环境变量以.env文件中的优先

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
)

In [ ]:
response = model.invoke("请翻译如下的汉字：你好世界")
print(response)

### 1.2 字典列表

举例1：

In [6]:
message = [
    {"role": "system", "content": "你是一个专业的数据老师"},
    {"role": "user", "content": "帮我解释一下什么是斐波那契数列"},
]
response = model.invoke(message)
print(response)

content='斐波那契数列（Fibonacci sequence）是一个非常有名的递推数列。核心定义很简单：\n\n> 从 0 和 1 开始，之后每一项都等于前两项之和。\n\n常见写法是：\n\n\\[\nF_0=0,\\quad F_1=1,\\quad F_n=F_{n-1}+F_{n-2}\\quad(n\\ge 2)\n\\]\n\n于是得到：\n\n\\[\n0,\\ 1,\\ 1,\\ 2,\\ 3,\\ 5,\\ 8,\\ 13,\\ 21,\\ 34,\\ 55,\\ 89,\\ 144,\\dots\n\\]\n\n也有人习惯从 1、1 开始写：\n\n\\[\n1,\\ 1,\\ 2,\\ 3,\\ 5,\\ 8,\\ 13,\\ 21,\\dots\n\\]\n\n它们本质一样，只是起点不同。\n\n---\n\n### 它怎么来的？\n\n它最早来自一个“兔子繁殖问题”：假设一对兔子一个月成熟，之后每个月生一对小兔子，且兔子不死，那么每个月的兔子对数就是斐波那契数列。\n\n所以它不是凭空规定的，而是从一种增长模型中自然产生的。\n\n---\n\n### 一个重要公式\n\n斐波那契数列的通项公式叫比内公式：\n\n\\[\nF_n=\\frac{\\varphi^n-(1-\\varphi)^n}{\\sqrt5}\n\\]\n\n其中\n\n\\[\n\\varphi=\\frac{1+\\sqrt5}{2}\\approx 1.6180339887\n\\]\n\n这个数就是著名的“黄金比例”。\n\n因此，当 \\(n\\) 很大时，斐波那契数列近似等于：\n\n\\[\nF_n\\approx \\frac{\\varphi^n}{\\sqrt5}\n\\]\n\n相邻两项的比值：\n\n\\[\n\\frac{F_{n+1}}{F_n}\n\\]\n\n会越来越接近黄金比例：\n\n\\[\n\\frac{1}{1}=1,\\quad \\frac{2}{1}=2,\\quad \\frac{3}{2}=1.5,\\quad \\frac{5}{3}\\approx1.666,\\quad \\frac{8}{5}=1.6,\\quad \\frac{13}{8}=1.625,\\dots\n\\]\n\n最后趋近于 \\

举例2：涉及多轮对话

In [ ]:
message = [
    {"role": "system", "content": "你是一个专业的数据老师"},
    {"role": "user", "content": "1 + 2 = ？"},
    {"role": "assistant", "content": "3"},
    {"role": "user", "content": "我刚才问了什么问题？"},
]
response = model.invoke(message)
print(response)

举例3：如果不传递历史，AI会”失忆“

In [7]:
messages1 = [
    {"role": "system", "content": "你是一个非常友好的AI助手"},
    {"role": "user", "content": "你好，我叫小明"},
]
# 第一次对话
response1 = model.invoke(messages1)
# 打印响应
print(f"AI的回复1：{response1.content}")
messages2 = [
    {"role": "user", "content": "我叫什么名字？"}
]
# 第二次对话
response2 = model.invoke(messages2)
# 打印响应
print(f"AI的回复2：{response2.content}")

AI的回复1：你好，小明！很高兴认识你 😊  
有什么我可以帮你的吗？无论是聊天、学习、写作还是解决问题，都可以告诉我。
AI的回复2：我不知道你的名字——你还没有告诉我。你可以告诉我，我就记住啦。


对比，传递记忆

In [8]:
conversation = [
    {"role": "system", "content": "你是一个非常友好的AI助手"},
    {"role": "user", "content": "你好，我叫小明"},
]
# 第一次对话
response1 = model.invoke(messages1)
# 打印响应
print(f"AI的回复1：{response1.content}")

# 添加记忆
conversation.append({"role": "assistant", "content": response1.content})
conversation.append({"role": "user", "content": "我叫什么名字？"})

# 第二次对话
response2 = model.invoke(conversation)
# 打印响应
print(f"AI的回复2：{response2.content}")


AI的回复1：你好，小明！很高兴认识你 😊 我是你的 AI 助手，有什么我可以帮你的吗？
AI的回复2：你叫小明呀 😊 这是你刚刚告诉我的。


### 1.3 消息对象列表
举例1：

In [ ]:

from langchain_core.messages.human import HumanMessage
from langchain_core.messages.system import SystemMessage

message = [
    SystemMessage(content="你是一个专业的数据老师"),
    HumanMessage(content="帮我解释一下什么是斐波那契数列"),
]
response = model.invoke(message)
print(response)

举例2：传递记忆

In [3]:

from langchain_core.messages.human import HumanMessage
from langchain_core.messages.system import SystemMessage
from langchain_core.messages.ai import AIMessage

messages = [
    SystemMessage(content="你是一个专业的数据老师"),
    HumanMessage(content="帮我解释一下什么是斐波那契数列"),
]
# 第一次对话
response1 = model.invoke(messages)
# 打印响应
print(f"AI的回复1：{response1.content}")

# 添加记忆
messages.append(AIMessage(content=response1.content))
messages.append(HumanMessage(content="我刚才问了什么问题？"), )

# 第二次对话
response2 = model.invoke(messages)
# 打印响应
print(f"AI的回复2：{response2.content}")

AI的回复1：斐波那契数列是一个非常经典的数列，核心规则很简单：

> **从第三项开始，每一项都等于前两项之和。**

最常见的定义是：

\[
F(0)=0,\quad F(1)=1
\]

\[
F(n)=F(n-1)+F(n-2),\quad n\ge 2
\]

所以数列是：

\[
0,\ 1,\ 1,\ 2,\ 3,\ 5,\ 8,\ 13,\ 21,\ 34,\ 55,\ 89,\ 144,\dots
\]

如果从 \(1,1\) 开始，也可以写成：

\[
1,\ 1,\ 2,\ 3,\ 5,\ 8,\ 13,\ 21,\dots
\]

两种写法本质一样，只是起点不同，工程或数学里通常约定清楚就行。

---

### 它的来历

斐波那契数列最早由意大利数学家斐波那契在《算盘书》中提出，用来描述一个“兔子繁殖问题”：

- 假设一对兔子每个月生一对新兔子；
- 新兔子出生后第二个月开始生育；
- 问每个月有多少对兔子？

结果就得到：

\[
1,\ 1,\ 2,\ 3,\ 5,\ 8,\ 13,\dots
\]

---

### 几个重要性质

**1. 相邻两项比值趋近黄金分割**

\[
\frac{F(n+1)}{F(n)} \to \varphi
\]

其中：

\[
\varphi=\frac{1+\sqrt{5}}{2}\approx 1.618
\]

这就是黄金比例。

**2. 有通项公式，叫比内公式**

\[
F(n)=\frac{\varphi^n-\psi^n}{\sqrt{5}}
\]

其中：

\[
\varphi=\frac{1+\sqrt{5}}{2},\quad \psi=\frac{1-\sqrt{5}}{2}
\]

虽然斐波那契数列是整数，但它可以用无理数表示出来。

**3. 增长非常快**

它近似按指数增长：

\[
F(n)\approx \frac{\varphi^n}{\sqrt{5}}
\]

所以 \(n\) 稍大，数值就会迅速变大。

---

### 在数据和计算机领域有什么用？

作为数据老师，我会特别强调它的几个应用场景：

1. **算法教学**  
   斐波那契数列是讲递归、动态规划、记忆化的经典例子。

2. **递归复杂度对比**  
   直接递归计算 \(

## invoke的返回值
举例1：

In [4]:
response = model.invoke([HumanMessage(content="2 + 3 * 2 = ?")])

In [5]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [6]:
print(response)

content='8' additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer simple arithmetic. Need follow order operations multiplication first 3*2=6 +2=8. Final concise.'} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 39, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 26, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 39}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'cb33c445-b33a-4f9a-b67e-d03f92ee1f91', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0f52d-d648-7090-8217-980306bfbd93-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 39, 'output_tokens': 28, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}, 'output_token_detai

美化输出

In [7]:
from rich import print as rprint

rprint(response)

AIMessage(
    content='8',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'We need answer simple arithmetic. Need follow order operations multiplication first 
3*2=6 +2=8. Final concise.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 28,
            'prompt_tokens': 39,
            'total_tokens': 67,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 26,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 39
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'cb33c445-b33a-4f9a-b67e-d03f92ee1f91',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0f52d-d648-7090-8217-980306bfbd93-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 39,
        'output_tokens': 28,
        'total_tokens': 67,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 26}
    }
)

举例2：

In [8]:
response = model.invoke("用一句话解释什么是 AI")

# 1. 获取回复内容
print("AI 回复:", response.content)

# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")

# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")

# 4. 获取消息 ID
print(f"消息 ID: {response.id}")


AI 回复: AI（人工智能）是让机器通过算法和数据模拟人类智能，从而完成学习、推理、决策和生成等任务的技术。
使用的模型: deepseek-flash
结束原因: stop
模型提供商：deepseek

输入 tokens: 35
输出 tokens: 85
总计 tokens: 120
消息 ID: lc_run--01a0f53b-2102-76c2-912e-5d3b5675f871-0
